# Workflow quickstart (3compartment)

A calibration study written as a **workflow**: one `calibration_workflow.json` whose steps run in
order, each one taking what it needs from the steps before it. This notebook builds that file
with `WorkflowBuilder`, one step per cell, and runs each step as it is added.

The file it writes is an ordinary workflow: `cuflynx-calibration-workflow` runs it from the
command line (on any number of MPI ranks) and CUFLynx's Workflow tab opens it. With the same
settings, seeds and rank count, all three give the same numbers, because they run the same file.

The steps, in the order they run:

```
screen  (sensitivity)      Sobol indices of two parameters against two features
  -> top  (reduction)      keeps the parameter the features are most sensitive to
  -> fit  (calibration)    fits it, with a genetic algorithm
  -> ia   (identifiability) Laplace approximation at the fit
emulate (emulation)        an emulator of the features (only with the [emulation] extra)
by_hand (set)              a value set by hand, with its reason
predict (sweep)            runs the calibrated target around the fitted value
```

Every builder call is used once: `add_operation`, `add_sensitivity`, `add_reduction`,
`add_calibration`, `add_identifiability`, `add_emulation`, `add_set`, `add_sweep`, then
`plan`, `run` (and `run(only=..., ranks=...)`), `status`, `result`, `plot_step`, `model`,
`write`, `WorkflowBuilder.load` and `pack_workflow`. The model is the 3-compartment
circulation in `resources/`; the longer tutorial is `generation_and_calibration.ipynb`.

In [ ]:
# Settings: the costly knobs. The defaults run in a few minutes; the CI test runs every cell of
# this notebook with smaller values.
warmup_time_s = 20.0          # s simulated before the logged second, to reach periodic steady state
sa_num_samples = 16           # Sobol samples (N); the analysis runs N (2 D + 2) simulations
ga_num_calls = 140            # genetic-algorithm model calls
emulator_train_samples = 32   # simulations the emulator is trained on

In [ ]:
import os
import shutil
import tempfile
from importlib.util import find_spec
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import SVG, Image, display

from libcuflynx.calibration_workflow import (WorkflowBuilder, module_array, pack_workflow,
                                             set_value)
from libcuflynx.utilities.obs_data_helpers import ObsDataCreator
from libcuflynx.utilities.paths import default_param_id_output_dir

this_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
CA_root = this_dir.parent.parent
resources_dir = CA_root / "resources"
# outputs go where the library puts them: the checkout, or $CUFLYNX_USER_DIR if set
param_id_output_dir = Path(default_param_id_output_dir())

# the measurement the model is calibrated to: one period of aortic pressure
gt = pd.read_csv(this_dir / "resources" / "aorta_pressure_temp.txt", sep="\t")
time_gt = gt["time_s"].to_numpy()
pressure_pa = gt["pressure_mmHg"].to_numpy() * 133.322  # mmHg -> Pa
obs_dt = round(float(time_gt[1] - time_gt[0]), 6)

## 1) A workflow and its settings

`target` is the model the results are merged into; a step's model is the target unless it names
another. The workflow's `settings` are what every step runs with. They are written into the
file, so every route that runs it uses the same solver and tolerances: the solver's longest step
and its tolerances are given here rather than left to each route's defaults.

In [ ]:
workflow_dir = param_id_output_dir / "workflows" / "3compartment_quickstart"
wf = WorkflowBuilder(
    "3compartment_quickstart",
    target=module_array(resources_dir / "3compartment_module_array.csv"),
    directory=workflow_dir,
    description="The workflow quickstart: screen, reduce, fit and predict on 3compartment.",
    settings={
        "solver": "CVODE_myokit",
        "solver_info": {"MaximumStep": 0.001, "rtol": 1e-6, "atol": 1e-8},
        "dt": obs_dt,  # output step; no larger than any obs_dt of the data
    },
)
print(wf)

## 2) The data: an obs_data built in code, with a feature of your own

`ObsDataCreator` builds the obs_data as a dict, as `CVS0DParamID` takes it. The builder writes
it to `data/` beside the workflow file when the workflow is written.

The protocol runs `warmup_time_s` of unlogged simulation (`pre_times`) before the logged period.
It has no `offline_pre_time`: that computes one steady state at the default parameter values and
starts every run from it, so a calibrated *initial* value, such as the blood volume
`global/q_lv_init` below, would be overwritten by that saved state.

`my_extra_feature` is an operation: a data item names it in `"operation"`, and its `operands` are
the function's arguments. `wf.add_operation` copies its source into the workflow's functions
file, so every rank, the command line and CUFLynx can use it. The file has the layout
CUFLynx's operation editor writes, so CUFLynx's Workflow tab and its operation editor open the
same files.

In [ ]:
def my_extra_feature(pressure, series_output=False):
    """The pressure half way through the logged period."""
    if series_output:  # the whole trace, for the automatic plots
        return pressure
    return pressure[len(pressure) // 2]

my_extra_feature.series_to_constant = True  # a series in, one value out
wf.add_operation(my_extra_feature)

obs_data_creator = ObsDataCreator()
obs_data_creator.add_protocol_info(pre_times=[warmup_time_s], sim_times=[[time_gt[-1]]],
                                   params_to_change={})
obs_data_creator.add_data_item({
    "data_item_name": "P aortic root mean", "trace_name_for_plotting": "P_{aoMean}",
    "operands": ["aortic_root/u"], "operation": "mean", "unit": "Pa",
    "value": np.mean(pressure_pa), "std": 200, "obs_dt": obs_dt,
})
obs_data_creator.add_data_item({
    "data_item_name": "P aortic root half time", "trace_name_for_plotting": "P_{aoHalf}",
    "operands": ["aortic_root/u"], "operation": "my_extra_feature", "unit": "Pa",
    "value": my_extra_feature(pressure_pa), "std": 200,
})

print(wf.funcs_source())  # the functions file the workflow names

In [ ]:
# the parameters to screen, with the bounds a fit may search
params_for_id = [
    {"vessel_name": "global", "param_name": "q_lv_init", "min": 200e-6, "max": 1500e-6,
     "name_for_plotting": "q_{sbv}"},
    {"vessel_name": "aortic_root", "param_name": "C", "min": 1e-9, "max": 5e-8,
     "name_for_plotting": "C_{ao}"},
]

## 3) `screen`: a Sobol sensitivity analysis

Each `add_*` call adds one step and checks it against the workflow schema at once. `wf.run()`
writes the files and runs every step that has not run, or whose inputs changed since; a step that
is done and current is not run again, so running a cell twice repeats no work. The seed fixes the
samples, so another run of the file draws the same ones.

In [ ]:
wf.add_sensitivity("screen", obs_data=obs_data_creator, params_for_id=params_for_id,
                   sa_options={"method": "sobol", "sample_type": "saltelli",
                               "num_samples": sa_num_samples, "seed": 1234})
wf.run()

for png in sorted(Path(wf.result("screen")["sensitivity_dir"]).glob("*Sobol_Heatmap.png")):
    display(Image(filename=str(png)))

## 4) `top`: keep the most influential parameter

A reduction step reads the Sobol indices of `screen` and writes the params_for_id of the
parameters it keeps, for a later step to fit: here the one with the largest total index, if that
index is above the threshold.

In [ ]:
wf.add_reduction("top", sensitivity_from="screen", method="sobol", n_params=1,
                 index_type="ST", criterion="max", threshold=0.05)
wf.run()
print("kept:", wf.result("top")["selected"])

## 5) `fit`: calibrate what `top` kept

`params_for_id_from="top"` fits the parameters the reduction kept, within their screening
bounds. The genetic algorithm gets a small population, so the tutorial runs quickly, and a
seed, which fixes the candidates it draws. `wf.plan()` shows what the run will
do before it does it, and `wf.plot_step` draws the best fit against the data.

In [ ]:
wf.add_calibration(
    "fit", obs_data=obs_data_creator, params_for_id_from="top",
    settings={"param_id_method": "genetic_algorithm",
              "optimiser_options": {"num_calls_to_function": ga_num_calls,
                                    "cost_convergence": 0.001, "max_patience": 10,
                                    "num_elite": 4, "num_survivors": 6,
                                    "num_mutations_per_survivor": 2, "num_cross_breed": 10,
                                    "seed": 1234}})
for step in wf.plan()["steps"]:
    print(f"{step['id']:8} {step['kind']:12} after {step['depends_on']}")

wf.run()
for plot_file in wf.plot_step("fit"):  # the images it wrote
    if plot_file.endswith(".png"):
        display(Image(filename=plot_file))
fit = wf.result("fit")
print("best fit:", fit["values"], "cost:", fit["best_cost"])

## 6) `ia`: identifiability at the fit

An identifiability step analyses parameters at the values a calibration found
(`result_from`). The Laplace approximation gives a standard deviation for each: a small one
means the data pin the parameter down, a large one that it is poorly identified.

In [ ]:
wf.add_identifiability("ia", obs_data=obs_data_creator, params_for_id=params_for_id,
                       ia_options={"method": "Laplace", "gradient_source": "FD"},
                       result_from="fit")
wf.run()

ia = wf.result("ia")
for name, at, std in zip(ia["parameters"], ia["at"], ia["laplace"]["std"]):
    print(f"{name}: {at:.4g} +- {std:.2g}")

## 7) `emulate`: an emulator of the features (optional)

An emulation step trains a fast surrogate of the features over the parameters `top` kept; a
calibration or sensitivity step with `emulator_from="emulate"` then evaluates it instead of the
model. It needs the optional `[emulation]` extra (`pip install "libcuflynx[emulation]"`), so
this cell skips it when that is not installed.

In [ ]:
if find_spec("autoemulate") is not None:
    wf.add_emulation("emulate", obs_data=obs_data_creator, params_for_id_from="top",
                     emulator_settings={"models": "PolynomialRegression",
                                        "num_train_samples": emulator_train_samples,
                                        "sample_type": "sobol", "random_seed": 0,
                                        "n_iter": 2, "n_splits": 2})
    wf.run()
else:
    print("autoemulate is not installed: no emulation step.")

## 8) `by_hand` and `predict`: a value set by hand, and a prediction on the calibrated target

A `set` step gives a parameter a value chosen by hand. It carries its reason into the workflow's
provenance, so a hand-set value is never mistaken for a calibrated one.

A sweep with `on_target=True` runs the target with every calibrated and set value, after all the
steps that give them; here at 0.9, 1 and 1.1 times the fitted blood volume. The obs_data gives
its protocol.

In [ ]:
wf.add_set("by_hand", [set_value("aortic_root/C", 1.5e-8, units="m6_per_J",
                                 reason="example: a compliance chosen by hand, which top "
                                        "did not keep for fitting")])
wf.add_sweep("predict", on_target=True, obs_data=obs_data_creator, outputs=["aortic_root/u"],
             parameters={"global/q_lv_init": {"factors": [0.9, 1.0, 1.1]}})
wf.run()

predict = np.load(wf.result("predict")["sweep_results"])
fig, ax = plt.subplots(figsize=(7, 4))
for point, trace in zip(predict["points"], predict["trace_0"]):
    ax.plot(predict["time_0"], trace[0], label=f"q_lv_init = {point[0]:.3g} m3")
ax.plot(time_gt, pressure_pa, "k--", label="data")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Aortic root pressure [Pa]")
ax.legend()
plt.show()

## 9) Change something, see what is stale, run again

Every step records the files it ran with. Add a step again with the same id and a changed value,
and `wf.status()` marks it, and every step after it that takes from it, as stale. `wf.run()` then
runs exactly those steps and keeps the others. (`wf.run(from_step="...")` runs a step and every
step after it whether current or not; `wf.run(force=True)` runs them all.)

In [ ]:
wf.add_set("by_hand", [set_value("aortic_root/C", 2.0e-8, units="m6_per_J",
                                 reason="example: a compliance chosen by hand, which top "
                                        "did not keep for fitting")])
print("stale:", [step for step, s in wf.status()["steps"].items() if s["stale"]])
wf.run()
print(wf)

## 10) The calibrated model

`wf.model("target")` generates the target with every value found so far (`wf.model("fit")`, a
step's own model with its values). Simulating it with the obs_data's protocol gives the
calibrated pressure.

In [ ]:
from libcuflynx.protocol_runners import ProtocolRunner

calibrated = wf.model("target")
for row in calibrated["calibrated"]:
    print(row["model_name"], row["value"], row.get("units", ""))

# the solver settings and protocol the steps ran with
runner = ProtocolRunner(calibrated["model_path"],
                        inp_data_dict={"dt": wf.settings["dt"],
                                       "solver_info": wf.settings["solver_info"]},
                        solver=wf.settings["solver"])
t_list, res_list, _ = runner.run_protocols(
    protocol_info=obs_data_creator.get_obs_data_dict()["protocol_info"],
    result_variables=["aortic_root/u"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(time_gt, pressure_pa, label="data")
ax.plot(t_list[0], res_list[0][0], label="calibrated target")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Aortic root pressure [Pa]")
ax.legend()
plt.tight_layout()
plt.savefig(wf.output_dir + "/calibrated_target.png")
plt.show()

## 11) The same workflow from the command line, on more ranks, and in CUFLynx

`wf.write()` returns the workflow file. It holds the steps, the settings, the data under `data/`
and the functions file, so it runs the same without this notebook:

- `cuflynx-calibration-workflow <file> --output-dir <dir>` runs it from a terminal (with
  `wf.output_dir` it continues from the results of this notebook), `mpiexec -n 4` before it
  runs it on four ranks, and `wf.run(ranks=4)` does the same from here;
- CUFLynx's **Workflow** tab opens the same file;
- `WorkflowBuilder.load(<file>)` reopens it here to add or change steps, and `pack_workflow`
  copies it with every file it needs into one folder, to send or archive.

A genetic algorithm's candidates depend on the number of ranks (each rank evaluates its own
batch), so compare runs made on the same number of ranks. A sweep's points do not, so the
prediction below, rerun on two ranks, gives the same pressures.

In [ ]:
path = wf.write()
command = f"cuflynx-calibration-workflow {path} --output-dir {wf.output_dir}"
print("from a terminal: ", command)
print("on four ranks:    mpiexec -n 4", command)

reopened = WorkflowBuilder.load(path)
print("steps of the reopened file:", reopened.steps)
pack_dir = tempfile.mkdtemp(prefix="3compartment_quickstart_packed_", dir=workflow_dir.parent)
packed = pack_workflow(path, pack_dir)
print("packed:", packed["workflow"])

if shutil.which("mpiexec"):
    one_rank = np.load(wf.result("predict")["sweep_results"])["trace_0"]
    wf.run(only="predict", ranks=2)  # run this step again, on two ranks
    two_ranks = np.load(wf.result("predict")["sweep_results"])["trace_0"]
    print("largest difference, one rank against two:", np.max(np.abs(two_ranks - one_rank)), "Pa")

display(SVG(filename=os.path.join(wf.output_dir, "provenance.svg")))